# Demo 1 — Offline basics (no API keys)

The lower-level building blocks of `dcpredictor` run fully offline. This notebook builds a small
synthetic route, generates a per-second **speed profile** with `DrivingCycleGenerator`, then computes
instantaneous **wheel power** and **diesel fuel rate** with the standalone longitudinal-vehicle-dynamics
(`lvd`) functions.

**Prerequisites**: `pip install -e .` in the repository root (conda env `dcp`). No API keys required.

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd

from dcpredictor.generators import DrivingCycleGenerator
from dcpredictor.utils.lvd import calculate_fuel_consumption_rate, calculate_wheel_power

## 1. Build a synthetic route

A route DataFrame has one row per waypoint: coordinates, the speed limits the driver model respects
(`MaxSpeed` / `BaseSpeed` / `TrafficSpeed`, in m/s), and an `Action` describing what happens at the
waypoint (`start`, `continue`, `turn`, `roundabout`, `arrive`, ...). In the end-to-end pipeline this
DataFrame comes from the HERE routing API; here we write it by hand.

In [ ]:
route_df = pd.DataFrame(
    {
        "Lat": [52.2000, 52.2050, 52.2120, 52.2180, 52.2240],
        "Lon": [0.1000, 0.1050, 0.1120, 0.1000, 0.0900],
        "MaxSpeed": [25.0, 25.0, 25.0, 20.0, 15.0],
        "BaseSpeed": [25.0, 25.0, 25.0, 20.0, 15.0],
        "TrafficSpeed": [25.0, 22.0, 25.0, 18.0, 15.0],
        "Action": ["start", "continue", "turn", "continue", "arrive"],
    }
)
route_df

## 2. Generate the speed profile

`generate_use_static_behaviour` simulates the vehicle along the route at a fixed time step
(default `dt = 1 s`): it looks ahead to the next constraining action (turn, arrival, ...), computes a
safe desired speed, and applies acceleration/deceleration limits.

In [ ]:
generator = DrivingCycleGenerator()
cycle_df = generator.generate_use_static_behaviour(route_df, datetime(2025, 2, 27, 9, 0, 0))

print(f"Time steps : {len(cycle_df)} (dt = 1 s)")
print(f"Distance   : {cycle_df['distance'].iloc[-1] / 1000:.2f} km")
print(f"Avg speed  : {cycle_df['speed'].mean() * 3.6:.1f} km/h")
print(f"Max speed  : {cycle_df['speed'].max() * 3.6:.1f} km/h")
cycle_df.head()

In [ ]:
fig, (ax_time, ax_dist) = plt.subplots(2, 1, figsize=(10, 6))

ax_time.plot(cycle_df["timestamp"], cycle_df["speed"] * 3.6)
ax_time.set_xlabel("Time")
ax_time.set_ylabel("Speed (km/h)")
ax_time.set_title("Speed profile vs time")
ax_time.grid(True)

ax_dist.plot(cycle_df["distance"] / 1000, cycle_df["speed"] * 3.6)
ax_dist.set_xlabel("Distance (km)")
ax_dist.set_ylabel("Speed (km/h)")
ax_dist.set_title("Speed profile vs distance")
ax_dist.grid(True)

plt.tight_layout()
plt.show()

## 3. Wheel power and fuel rate (flat road)

`calculate_wheel_power` implements `P = (F_roll + F_grade + F_aero + F_accel) * v`;
`calculate_fuel_consumption_rate` converts wheel power to a diesel fuel rate through the driveline and
engine efficiencies. Here we assume a flat road (`gradient_degrees = 0`); the end-to-end pipeline
(Demo 2) uses the real elevation profile instead.

In [ ]:
MASS_KG = 5000.0

cycle_df["wheel_power_kW"] = cycle_df.apply(
    lambda row: calculate_wheel_power(
        mass_kg=MASS_KG,
        gradient_degrees=0.0,
        velocity_mps=row["speed"],
        acceleration_mps2=row["acc"],
    )
    / 1000.0,
    axis=1,
)
cycle_df["fuel_rate_L_hr"] = cycle_df.apply(
    lambda row: calculate_fuel_consumption_rate(row["wheel_power_kW"] * 1000.0)["rate"],
    axis=1,
)

fig, (ax_p, ax_f) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax_p.plot(cycle_df["distance"] / 1000, cycle_df["wheel_power_kW"], color="tab:blue")
ax_p.set_ylabel("Wheel power (kW)")
ax_p.set_title("Wheel power and fuel rate vs distance (flat road)")
ax_p.grid(True)

ax_f.plot(cycle_df["distance"] / 1000, cycle_df["fuel_rate_L_hr"], color="tab:green")
ax_f.set_xlabel("Distance (km)")
ax_f.set_ylabel("Fuel rate (L/hr)")
ax_f.grid(True)

plt.tight_layout()
plt.show()

idx = cycle_df["speed"].idxmax()
print(
    f"At max speed ({cycle_df.loc[idx, 'speed'] * 3.6:.1f} km/h): "
    f"{cycle_df.loc[idx, 'wheel_power_kW']:.1f} kW, "
    f"{cycle_df.loc[idx, 'fuel_rate_L_hr']:.2f} L/hr"
)

## Next steps

- **Demo 2** (`predict_end_to_end.ipynb`) — the full pipeline: HERE route → speed profile →
  SRF elevation/gradient → energy/fuel, via `DutyCyclePredictor.predict()` (needs API keys).
- **Demo 3** (`predict_vs_measured_leg.ipynb`) — validate a prediction against a measured GPS trip leg
  from `../tests/data/`.